In [3]:
# Cell 1: Load processed data
import pickle
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score)
from xgboost import XGBClassifier

with open('../data/processed_data.pkl', 'rb') as f:
    X_train, X_test, y_train, y_test, feature_names = pickle.load(f)

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# Cell 2: Train 3 models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    print(f"\n{'='*40}")
    print(f"📊 {name}")
    print(f"Accuracy: {results[name]['accuracy']:.4f}")
    print(f"ROC-AUC:  {results[name]['roc_auc']:.4f}")



In [ ]:
# Cell 3: Model comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy & ROC-AUC comparison
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] for m in model_names]
roc_aucs = [results[m]['roc_auc'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35
bars1 = axes[0].bar(x - width/2, accuracies, width, label='Accuracy', color='#3498db')
bars2 = axes[0].bar(x + width/2, roc_aucs, width, label='ROC-AUC', color='#e74c3c')
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=15)
axes[0].set_ylim(0.7, 0.95)
axes[0].legend()
axes[0].bar_label(bars1, fmt='%.3f', padding=3)
axes[0].bar_label(bars2, fmt='%.3f', padding=3)

# ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves Comparison', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4: Best model - Feature Importance
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']
print(f"\n🏆 Best model: {best_model_name}")

if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(10, 8))
    sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
    plt.title(f'Top 15 Feature Importances ({best_model_name})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()